In [ ]:
import glob
import gzip
import io
import itertools
import json
import math
import os
import pickle
import re
import subprocess
import warnings
from pathlib import Path
from statistics import median

import boto3
import circlify as circ
import matplotlib
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import matplotlib.patches as patches
import matplotlib.patheffects as PathEffects
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import umap as _umap_lib
from IPython.display import display
from matplotlib import patheffects
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_hex, to_rgba
from scipy.spatial.distance import pdist, squareform
from scipy.stats import mannwhitneyu, norm, pearsonr
from sklearn.decomposition import PCA
from sklearn.decomposition import PCA as _PCA2
from sklearn.manifold import TSNE as _TSNE
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler as _SS
from statsmodels.stats.multitest import multipletests
from supervenn import supervenn
from tqdm import tqdm
from tqdm.notebook import tqdm as tqdm_notebook

%config InlineBackend.figure_format = 'png'
plt.rcParams["pdf.fonttype"] = "truetype"
plt.rcParams["figure.dpi"] = 200

plt.rcParams.update(
    {
        "svg.fonttype": "none",
        "font.size": 7.5,
        "axes.titlesize": 7.5,
        "axes.labelsize": 7.5,
        "xtick.labelsize": 7.5,
        "ytick.labelsize": 7.5,
        "legend.fontsize": 7.5,
        "figure.titlesize": 7.5,
    }
)

sns.set_style("ticks")
sns.axes_style("white")

# Cohort assembly originally ran against an internal cohort database, configured
# here with an API key. The public pipeline reads prepared tables directly, so no
# client and no credential is needed.

import csv

import matplotlib.patches as patches
import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
import pyreadr
import rpy2.robjects as ro
from biomart import BiomartServer
from matplotlib.path import Path
from neuroCombat import neuroCombat
from rpy2.robjects import numpy2ri, pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr
from rpy2.robjects.vectors import (
    BoolVector,
    DataFrame,
    FloatVector,
    IntVector,
    ListVector,
    Matrix,
    StrVector,
)
from scipy.interpolate import interp1d
from sklearn.cluster import AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# S3 scripts

In [ ]:
files = !aws s3 ls s3://$FL_S3_BUCKET/FL_batch_correction/metrics/H_affymetrix_extended

In [ ]:
for file in files:
    file_processed = file.split(" ")[-1]
    if "shambhala" not in file_processed:
        print(
            f"aws s3 rm s3://$FL_S3_BUCKET/FL_batch_correction/metrics/{file_processed}"
        )

# Setup

## Palettes

In [ ]:
# ── §5.0 Canonical project palettes (from all_cohorts_assembly.ipynb) ─────────
# Definitive color mappings for the entire dissertation.
# Use for all sample-level scatter plots in §5.

lymphoma_ontogeny_palette = {
    "Bone_marrow_CD19+": "blue",
    "Immature": "#08519c",
    "Naive": "#4292c6",
    "GC": "#006d2c",
    "Centroblast": "#41ab5d",
    "Centrocyte": "#a1d99b",
    "B_cells": "lawngreen",
    "MZ": "olive",
    "Memory": "#00ced1",
    "Plasmablast": "#5f9ea0",
    "Plasma": "grey",
    "Follicular_Lymphoma": "#6a0dad",
    "Follicular_Lymphoma_GCB": "#9400d3",
    "Diffuse_Large_B_Cell_Lymphoma_ABC": "#ff4500",
    "Diffuse_Large_B_Cell_Lymphoma_GCB": "#d62728",
    "Diffuse_Large_B_Cell_Lymphoma": "#8b0000",
    "High_Grade_B_Cell_Lymphoma": "#ff00ff",
    "Burkitt_Lymphoma": "orange",
}

rna_batch_palette = {
    "RNASeq_FF_Total": "#1f77b4",
    "RNASeq_FF_PolyA": "#17becf",
    "RNASeq_FFPE_Exome_capture": "#9467bd",
    "RNASeq_FF_rRNADepletion": "#000080",
    "RNASeq_FFPE_PolyA": "#8c564b",
    "RNASeq_FF_Unknown": "#bcbd22",
    "RNASeq_FF_Exome_capture": "#2ca02c",
    "GPL96+97_FF_Unknown": "#d62728",
    "GPL570_FF_Unknown": "#ff7f0e",
    "GPL570_Unknown_Unknown": "#ffbb78",
    "GPL570_FFPE_Unknown": "#dbdb8d",
    "GPL96_FF_Unknown": "#f7b6d2",
    "GPL96_FFPE_Unknown": "#e377c2",
    "GPL6244_FF_Unknown": "#e9967a",
    "GPL6244_FFPE_Unknown": "#fa8072",
    "GPL13938_FFPE_Unknown": "#800000",
    "GPL1708_FF_Unknown": "#7f7f7f",
    "GPL14951_FFPE_Unknown": "#c49c94",
    "GPL17077_FF_Unknown": "#c5b0d5",
    "GPL13158_FF_Unknown": "#ad494a",
    "GPL17586_FF_Unknown": "#008b8b",
    "GPL17047_FF_Unknown": "#4682b4",
    "GPL887_FFPE_Unknown": "#b0c4de",
    "GPL10739_FF_Unknown": "#9edae5",
    "GPL8432_FFPE_Unknown": "#006400",
    "GPL20188_FF_Unknown": "#556b2f",
    "GPL23541_FF_Unknown": "#8fbc8f",
    "GPL16686_FF_Unknown": "#333333",
    "GPL26356_FF_Unknown": "#a0522d",
}

platform_palette = {
    "RNASeq": "#4B0082",
    "RNAseq": "#4B0082",
    "Kassandra": "#6A5ACD",
    "GPL96+97": "#FF4500",
    "GPL96+GPL97": "#FF4500",
    "GPL570": "#FF8C00",
    "GPL96": "#B22222",
    "GPL6244": "#E9967A",
    "GPL13938": "#CD5C5C",
    "A-GEOD-19803": "#A52A2A",
    "GPL1708": "#008080",
    "GPL14951": "#4682B4",
    "GPL17077": "#00CED1",
    "GPL13158": "#1E90FF",
    "GPL17586": "#20B2AA",
    "GPL17047": "#5F9EA0",
    "GPL887": "#B0C4DE",
    "GPL10739": "#2E8B57",
    "GPL10739+GPL19251": "#3CB371",
    "GPL20188": "#6B8E23",
    "GPL23541": "#8FBC8F",
    "GPL8432": "#006400",
    "GPL16686": "#8B4513",
    "GPL26356": "#A0522D",
    float("nan"): "#D3D3D3",
    None: "#D3D3D3",
}

strat_pal = {
    "S0_no_removal": (0.6313725490196078, 0.788235294117647, 0.9568627450980393),
    "A_confirmed_bad": (1.0, 0.7058, 0.5098),
    "B_extended_bad": (0.5529, 0.8980, 0.6313),
    "C_rnaseq_only": (1.0, 0.6235, 0.6078),
    "D_malignant_only": (0.8156, 0.7333, 1.0),
    "E1_iterative_r1": (0.8705, 0.7333, 0.6078),
    "E2_iterative_r2": (0.9803, 0.6901, 0.8941),
    "E3_iterative_r3": (0.8117, 0.8117, 0.8117),
    "F_microarray_only": (1.0, 0.9960, 0.6392),
    "G_affymetrix_only": (0.7254, 0.9490, 0.9411),
    "H_affymetrix_extended": (0.741, 0.718, 0.420),
    "I_rare_batches_removed": "#b095c5",  # previous version was '#7b4f9e',
    "J_ff_only": "#7ecbc4",  # soft teal — fresh frozen
    "K_ffpe_only": "#f2c27f",  # warm amber — FFPE
}

## Ann and expression files download

In [ ]:
s3 = boto3.client("s3")
body = s3.get_object(
    Bucket="$FL_S3_BUCKET",
    Key="FL_batch_correction/exp/comb_exp.tsv",#"FL_batch_correction/exp/S0_no_removal__strict__01_raw__post0.tsv.gz", 
)["Body"].read()

#comb_exp = pd.read_csv(io.BytesIO(gzip.decompress(body)), sep="\t", index_col=0)
comb_exp = pd.read_csv(io.BytesIO(body), sep="\t", index_col=0)
comb_exp

In [ ]:
pd.DataFrame((~comb_exp.isna()).sum(axis=1), columns=['Number of genes with expression']).to_csv("Supplementary File 2.csv")

In [ ]:
comb_exp

In [ ]:
s3 = boto3.client("s3")
body = s3.get_object(
    Bucket="$FL_S3_BUCKET",
    Key="FL_batch_correction/prepared/S0_no_removal__knn__ann.tsv.gz",
)["Body"].read()

comb_ann = pd.read_csv(io.BytesIO(gzip.decompress(body)), sep="\t", index_col=0)

batch_to_plaform_group = {
    "RNASeq_FF_PolyA": "Illumina NGS",
    "GPL570_Unknown_Unknown": "Affymetrix Microarray",
    "GPL570_FF_Unknown": "Affymetrix Microarray",
    "RNASeq_FFPE_Exome_capture": "Illumina NGS",
    "GPL14951_FFPE_Unknown": "Illumina Microarray",
    "GPL570_FFPE_Unknown": "Affymetrix Microarray",
    "GPL96+97_FF_Unknown": "Affymetrix Microarray",
    "GPL13938_FFPE_Unknown": "Illumina Microarray",
    "GPL96_FF_Unknown": "Affymetrix Microarray",
    "GPL6244_FF_Unknown": "Affymetrix Microarray",
    "GPL8432_FFPE_Unknown": "Illumina Microarray",
    "RNASeq_FF_rRNADepletion": "Illumina NGS",
    "GPL96_FFPE_Unknown": "Affymetrix Microarray",
    "RNASeq_FF_Unknown": "Illumina NGS",
    "GPL20188_FF_Unknown": "Affymetrix Microarray",
    "GPL16686_FF_Unknown": "Affymetrix Microarray",
    "GPL13158_FF_Unknown": "Affymetrix Microarray",
    "GPL17586_FF_Unknown": "Affymetrix Microarray",
    "GPL1708_FF_Unknown": "Agilent Microarray",
    "GPL23541_FF_Unknown": "Affymetrix Microarray",
    "RNASeq_FFPE_PolyA": "Illumina NGS",
    "RNASeq_FF_Total": "Illumina NGS",
    "GPL17077_FF_Unknown": "Agilent Microarray",
    "GPL887_FFPE_Unknown": "Agilent Microarray",
    "GPL26356_FF_Unknown": "Affymetrix Microarray",
    "GPL17047_FF_Unknown": "Affymetrix Microarray",
    "GPL6244_FFPE_Unknown": "Affymetrix Microarray",
    "GPL10739_FF_Unknown": "Affymetrix Microarray",
    "RNASeq_FF_Exome_capture": "Illumina NGS",
}

comb_ann["Platform_group"] = comb_ann.RNA_BATCH.map(batch_to_plaform_group)

comb_ann["Platform_group"].fillna("NA").value_counts()

comb_ann

In [ ]:
comb_ann.AUTHOR.value_counts()

In [ ]:
pd.set_option('display.max_rows', 100)
pd.DataFrame(comb_ann.COHORT_LABEL.value_counts())

In [ ]:
# columns_to_save = [col for col in comb_ann.isna().sum().sort_values().index[:10] if col != 'AUTHOR']

columns_to_save = [
    "Diagnosis_cell_type_unified",
    "Diagnosis_with_coo",
    "Major_group",
    "RNA_BATCH",
    "COHORT_LABEL",
    "TUMOR_NORMAL",
    "RNASEQ_SOURCE",
    "PLATFORM_RNA",
    "Platform_group",
    'NEO_CHEMO_THERAPY', 
    'GENDER',
    'PUBLIC_REPOSITORY_ID',
    'OS',
    'OS_FLAG'
]
comb_ann[columns_to_save].to_csv("Supplementary File 1.csv")

In [ ]:
comb_ann.isna().sum().sort_values().index[:20]

In [ ]:
print(comb_ann.isna().sum().sort_values().to_string())

# Introductory figures

## Bubble chart of samples by their batches amd biology

In [ ]:
comb_ann[
    [
        "RNA_BATCH",
        "TUMOR_NORMAL",
        "Major_group",
        "Diagnosis_cell_type_unified",
        "COHORT_LABEL",
        "Platform_group",
    ]
]

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
sns.barplot(comb_ann.RNA_BATCH.value_counts(), ax=ax)
ax.set_xticklabels(comb_ann.RNA_BATCH.value_counts().index, rotation=90, ha="right")
fig.savefig("./figures/trial_barplot.svg", format="svg", bbox_inches="tight")
plt.show()

In [ ]:
def generate_strict_hex_grid_points(N, spacing=1):
    """Generates N points on a strict horizontal hexagonal grid starting from center."""
    dx = spacing
    dy = spacing * math.sqrt(3) / 2.0
    side = math.ceil(math.sqrt(N)) + 5
    points = []

    for row in range(-side, side):
        for col in range(-side, side):
            x = col * dx + (row % 2) * (dx / 2.0)
            y = row * dy
            points.append((x, y))

    points.sort(key=lambda p: p[0] ** 2 + p[1] ** 2)
    return points[:N]


def draw_gradient_halo(ax, x, y, max_radius, zorder=8):
    """
    Draws a vector-friendly radial gradient using stacked concentric circles.
    This creates a smooth fade from solid white in the center to transparent at the edge.
    """
    num_steps = 30
    for i in range(num_steps):
        # Shrink the radius with each step
        r = max_radius * (1 - (i / num_steps))
        # Stacked low-alpha patches naturally accumulate to 1.0 (100% opacity) in the center
        circle = plt.Circle(
            (x, y),
            radius=r,
            facecolor="white",
            edgecolor="none",
            alpha=0.08,
            zorder=zorder,
        )
        ax.add_patch(circle)


def create_depmap_style_bubble_chart(
    df, rna_palette, spacing_intra_platform=2.0, spacing_inter_platform=10.0
):

    pack_data = []
    for platform, plat_df in df.groupby("Platform_group"):
        children = [
            {"id": batch, "datum": len(batch_df)}
            for batch, batch_df in plat_df.groupby("RNA_BATCH")
        ]
        if children:
            parent_datum = sum(child["datum"] for child in children)
            pack_data.append(
                {"id": platform, "datum": parent_datum, "children": children}
            )

    circles = circ.circlify(pack_data, show_enclosure=False)

    nodes = []
    for platform, plat_df in df.groupby("Platform_group"):
        for batch, batch_df in plat_df.groupby("RNA_BATCH"):
            N = len(batch_df)
            R_tight = math.sqrt((N * 0.866) / math.pi)

            cx, cy = 0.0, 0.0
            for c in circles:
                if c.ex["id"] == batch:
                    cx, cy = c.x * 100, c.y * 100
                    break

            nodes.append(
                {
                    "id": batch,
                    "platform": platform,
                    "N": N,
                    "R_tight": R_tight,
                    "x": cx,
                    "y": cy,
                }
            )

    print(
        f"Running physics layout (Intra-spacing: {spacing_intra_platform}, Inter-spacing: {spacing_inter_platform})..."
    )
    for _ in range(400):
        for n in nodes:
            n["x"] -= n["x"] * 0.015
            n["y"] -= n["y"] * 0.015

        platform_centroids = {}
        for p in df["Platform_group"].unique():
            p_nodes = [n for n in nodes if n["platform"] == p]
            if p_nodes:
                c_x = sum(n["x"] for n in p_nodes) / len(p_nodes)
                c_y = sum(n["y"] for n in p_nodes) / len(p_nodes)
                platform_centroids[p] = (c_x, c_y)

        for n in nodes:
            c_x, c_y = platform_centroids[n["platform"]]
            n["x"] += (c_x - n["x"]) * 0.05
            n["y"] += (c_y - n["y"]) * 0.05

        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                n1, n2 = nodes[i], nodes[j]
                dx = n1["x"] - n2["x"]
                dy = n1["y"] - n2["y"]
                dist = math.hypot(dx, dy)

                min_dist = n1["R_tight"] + n2["R_tight"]

                if n1["platform"] == n2["platform"]:
                    min_dist += spacing_intra_platform
                else:
                    min_dist += spacing_inter_platform

                if dist < min_dist and dist > 0:
                    overlap = min_dist - dist
                    push_x = (dx / dist) * overlap * 0.5
                    push_y = (dy / dist) * overlap * 0.5
                    n1["x"] += push_x
                    n1["y"] += push_y
                    n2["x"] -= push_x
                    n2["y"] -= push_y

    all_x, all_y, all_c = [], [], []

    for i, n in enumerate(nodes):
        strict_points = generate_strict_hex_grid_points(n["N"], spacing=1.2)

        angle_degrees = (i * 43) % 360
        theta = math.radians(angle_degrees)
        cos_t, sin_t = math.cos(theta), math.sin(theta)

        color = rna_palette.get(n["id"], "#cccccc")

        for px, py in strict_points:
            nx = px * cos_t - py * sin_t
            ny = px * sin_t + py * cos_t
            all_x.append(nx + n["x"])
            all_y.append(ny + n["y"])
            all_c.append(color)

    fig, ax = plt.subplots(figsize=(15, 7.5), dpi=300)
    ax.set_aspect("equal")
    ax.axis("off")

    min_x, max_x = min(all_x), max(all_x)
    min_y, max_y = min(all_y), max(all_y)
    pad = (max_x - min_x) * 0.05
    ax.set_xlim(min_x - pad, max_x + pad)
    ax.set_ylim(min_y - pad, max_y + pad)

    fig_width_inches = fig.get_size_inches()[0]
    ax_width_points = (fig_width_inches * 0.95) * 72
    data_width = (max_x + pad) - (min_x - pad)

    pts_per_unit = ax_width_points / data_width
    marker_size = (0.55 * pts_per_unit) ** 2

    # Draw the data points (zorder=1)
    ax.scatter(
        all_x, all_y, s=marker_size, c=all_c, edgecolors="none", alpha=0.9, zorder=1
    )

    # -------------------------------------------------------------
    # Render Platform Group Text & Halos
    # -------------------------------------------------------------
    for p in df["Platform_group"].unique():
        p_nodes = [n for n in nodes if n["platform"] == p]
        if not p_nodes:
            continue

        total_N = sum(n["N"] for n in p_nodes)
        c_x = sum(n["x"] * n["N"] for n in p_nodes) / total_N
        c_y = sum(n["y"] * n["N"] for n in p_nodes) / total_N

        platform_text = p.replace("_", "\n")
        font_size = max(12, math.sqrt(total_N) * 0.4)

        # Determine the radius of the gradient based on the cluster size
        halo_radius = math.sqrt(total_N) * 0.45

        # Draw the gradient halo underneath the text (zorder=9.5)
        # draw_gradient_halo(ax, c_x, c_y, halo_radius, zorder=9.5)

        # Draw the text on top (zorder=10)
        ax.text(
            c_x,
            c_y,
            platform_text,
            ha="center",
            va="center",
            fontsize=font_size,
            weight="bold",
            zorder=10,
        )

    # -------------------------------------------------------------
    # Render RNA BATCH Text & Halos
    # -------------------------------------------------------------
    for n in nodes:
        if n["N"] > 0:
            batch_text = n["id"].replace("_", "\n")

            # Text coordinates
            t_x = n["x"]
            t_y = n["y"]  # - (n['R_tight'] * 0.5)

            # Make the halo slightly smaller than the sub-cluster itself
            halo_radius = n["R_tight"] * 0.55

            # Draw the gradient halo underneath the text (zorder=8.5)
            draw_gradient_halo(ax, t_x, t_y, halo_radius, zorder=8.5)

            # Draw the text on top (zorder=9)
            ax.text(
                t_x,
                t_y,
                batch_text,
                ha="center",
                va="center",
                fontsize=7.5,
                weight="bold",
                zorder=9,
            )

    plt.tight_layout()
    print("Saving files...")

    import os

    os.makedirs("./figures", exist_ok=True)  # Ensure directory exists

    fig.savefig(
        "./figures/comb_ann_platform_groups_bubble_chart.svg",
        format="svg",
        bbox_inches="tight",
        transparent=True,
    )
    fig.savefig(
        "./figures/comb_ann_platform_groups_bubble_chart.png",
        format="png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


# Run the plotting function
create_depmap_style_bubble_chart(
    comb_ann, rna_batch_palette, spacing_intra_platform=5.0, spacing_inter_platform=11.0
)

In [ ]:
def generate_candidate_hex_points(N, extra_factor=3.0, spacing=1.2):
    """
    Generates an excess pool of points on a strict horizontal hexagonal grid.
    extra_factor ensures we have backup coordinates if edge points collide with other clusters.
    """
    dx = spacing
    dy = spacing * math.sqrt(3) / 2.0

    target_points = int(N * extra_factor)
    side = math.ceil(math.sqrt(target_points)) + 5
    points = []

    for row in range(-side, side):
        for col in range(-side, side):
            x = col * dx + (row % 2) * (dx / 2.0)
            y = row * dy
            points.append((x, y))

    # Sort from center outward
    points.sort(key=lambda p: p[0] ** 2 + p[1] ** 2)
    return points[:target_points]


def draw_gradient_halo(ax, x, y, max_radius, zorder=8):
    """
    Draws a vector-friendly radial gradient using stacked concentric circles.
    This creates a smooth fade from solid white in the center to transparent at the edge.
    """
    num_steps = 30
    for i in range(num_steps):
        r = max_radius * (1 - (i / num_steps))
        circle = plt.Circle(
            (x, y),
            radius=r,
            facecolor="white",
            edgecolor="none",
            alpha=0.08,
            zorder=zorder,
        )
        ax.add_patch(circle)


def create_depmap_style_bubble_chart(
    df,
    rna_palette,
    spacing_intra_platform=5.0,
    spacing_inter_platform=11.0,
    figsize=(10, 5),  # <-- Added figsize
    dot_size_scale=0.55,
):  # <-- Added dot size param

    pack_data = []
    for platform, plat_df in df.groupby("Platform_group"):
        children = [
            {"id": batch, "datum": len(batch_df)}
            for batch, batch_df in plat_df.groupby("RNA_BATCH")
        ]
        if children:
            parent_datum = sum(child["datum"] for child in children)
            pack_data.append(
                {"id": platform, "datum": parent_datum, "children": children}
            )

    circles = circ.circlify(pack_data, show_enclosure=False)

    nodes = []
    HEX_SPACING = 1.2

    for platform, plat_df in df.groupby("Platform_group"):
        for batch, batch_df in plat_df.groupby("RNA_BATCH"):
            N = len(batch_df)
            # Area of hex grid scaled by spacing^2
            R_tight = math.sqrt((N * 0.866 * (HEX_SPACING**2)) / math.pi)

            cx, cy = 0.0, 0.0
            for c in circles:
                if c.ex["id"] == batch:
                    cx, cy = c.x * 100, c.y * 100
                    break

            nodes.append(
                {
                    "id": batch,
                    "platform": platform,
                    "N": N,
                    "R_tight": R_tight,
                    "x": cx,
                    "y": cy,
                }
            )

    print(
        f"Running physics layout (Intra-spacing: {spacing_intra_platform}, Inter-spacing: {spacing_inter_platform})..."
    )
    for _ in range(400):
        for n in nodes:
            n["x"] -= n["x"] * 0.015
            n["y"] -= n["y"] * 0.015

        platform_centroids = {}
        for p in df["Platform_group"].unique():
            p_nodes = [n for n in nodes if n["platform"] == p]
            if p_nodes:
                c_x = sum(n["x"] for n in p_nodes) / len(p_nodes)
                c_y = sum(n["y"] for n in p_nodes) / len(p_nodes)
                platform_centroids[p] = (c_x, c_y)

        for n in nodes:
            c_x, c_y = platform_centroids[n["platform"]]
            n["x"] += (c_x - n["x"]) * 0.05
            n["y"] += (c_y - n["y"]) * 0.05

        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                n1, n2 = nodes[i], nodes[j]
                dx = n1["x"] - n2["x"]
                dy = n1["y"] - n2["y"]
                dist = math.hypot(dx, dy)

                min_dist = n1["R_tight"] + n2["R_tight"]

                if n1["platform"] == n2["platform"]:
                    min_dist += spacing_intra_platform
                else:
                    min_dist += spacing_inter_platform

                if dist < min_dist and dist > 0:
                    overlap = min_dist - dist
                    push_x = (dx / dist) * overlap * 0.5
                    push_y = (dy / dist) * overlap * 0.5
                    n1["x"] += push_x
                    n1["y"] += push_y
                    n2["x"] -= push_x
                    n2["y"] -= push_y

    # =========================================================================
    # DOT PLACEMENT & EDGE MORPHING (Continents & Seas)
    # =========================================================================

    # 1. Pre-generate and transform excess candidate pools for every cluster
    cluster_candidates = {}
    for i, n in enumerate(nodes):
        candidates = generate_candidate_hex_points(
            n["N"], extra_factor=3.0, spacing=HEX_SPACING
        )

        angle_degrees = (i * 43) % 360
        theta = math.radians(angle_degrees)
        cos_t, sin_t = math.cos(theta), math.sin(theta)

        transformed_pool = []
        for px, py in candidates:
            nx = px * cos_t - py * sin_t + n["x"]
            ny = px * sin_t + py * cos_t + n["y"]
            transformed_pool.append((nx, ny))

        cluster_candidates[i] = transformed_pool

    # 2. Setup Spatial Hash for fast collision detection
    collision_radius = HEX_SPACING * 0.95
    spatial_hash = {}

    def check_collision(x, y, cid):
        """Checks if coordinate is too close to a dot belonging to a DIFFERENT cluster"""
        cx, cy = int(x / collision_radius), int(y / collision_radius)
        # Check surrounding cells
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                cell = (cx + dx, cy + dy)
                if cell in spatial_hash:
                    for px, py, pcid in spatial_hash[cell]:
                        if pcid != cid:  # Only avoid OTHER clusters
                            if math.hypot(px - x, py - y) < collision_radius:
                                return True
        return False

    # 3. Round-Robin Placement
    all_x, all_y, all_c = [], [], []
    placed_counts = {i: 0 for i in range(len(nodes))}
    candidate_indices = {i: 0 for i in range(len(nodes))}

    # Loop until every cluster has placed exactly N dots
    while any(placed_counts[i] < nodes[i]["N"] for i in range(len(nodes))):
        for i, n in enumerate(nodes):
            if placed_counts[i] < n["N"]:
                placed = False

                # Check candidates until we find one that doesn't overlap an adjacent cluster
                while candidate_indices[i] < len(cluster_candidates[i]) and not placed:
                    px, py = cluster_candidates[i][candidate_indices[i]]
                    candidate_indices[i] += 1

                    if not check_collision(px, py, i):
                        # Accept position
                        all_x.append(px)
                        all_y.append(py)
                        all_c.append(rna_palette.get(n["id"], "#cccccc"))
                        placed_counts[i] += 1

                        # Add to spatial hash
                        cx, cy = int(px / collision_radius), int(py / collision_radius)
                        if (cx, cy) not in spatial_hash:
                            spatial_hash[(cx, cy)] = []
                        spatial_hash[(cx, cy)].append((px, py, i))

                        placed = True

    # =========================================================================

    # Setup Plot using dynamic figsize
    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    ax.set_aspect("equal")
    ax.axis("off")

    min_x, max_x = min(all_x), max(all_x)
    min_y, max_y = min(all_y), max(all_y)
    pad = (max_x - min_x) * 0.05
    ax.set_xlim(min_x - pad, max_x + pad)
    ax.set_ylim(min_y - pad, max_y + pad)

    # Scale dots using the new dot_size_scale parameter
    fig_width_inches = fig.get_size_inches()[0]
    ax_width_points = (fig_width_inches * 0.95) * 72
    data_width = (max_x + pad) - (min_x - pad)

    pts_per_unit = ax_width_points / data_width
    marker_size = (dot_size_scale * pts_per_unit) ** 2

    # Draw the data points (zorder=1)
    ax.scatter(
        all_x, all_y, s=marker_size, c=all_c, edgecolors="none", alpha=0.9, zorder=1
    )

    # -------------------------------------------------------------
    # Render Platform Group Text & Halos
    # -------------------------------------------------------------
    for p in df["Platform_group"].unique():
        p_nodes = [n for n in nodes if n["platform"] == p]
        if not p_nodes:
            continue

        total_N = sum(n["N"] for n in p_nodes)
        c_x = sum(n["x"] * n["N"] for n in p_nodes) / total_N
        c_y = sum(n["y"] * n["N"] for n in p_nodes) / total_N

        platform_text = p.replace("_", "\n")
        # font_size = max(10, math.sqrt(total_N) * 0.2)

        ax.text(
            c_x,
            c_y,
            platform_text,
            ha="center",
            va="center",
            fontsize=7.5,
            weight="bold",
            zorder=10,
        )

    # -------------------------------------------------------------
    # Render RNA BATCH Text & Halos
    # -------------------------------------------------------------
    for n in nodes:
        if n["N"] > 50:
            current_number = n["N"]
            batch_text = n["id"].replace("_", "\n") + f"\n({str(current_number)})"

            t_x = n["x"]
            t_y = n["y"]

            # Make the halo slightly smaller than the sub-cluster itself
            halo_radius = n["R_tight"] * 0.75
            draw_gradient_halo(ax, t_x, t_y, halo_radius, zorder=8.5)

            ax.text(
                t_x, t_y, batch_text, ha="center", va="center", fontsize=7.5, zorder=9
            )

    plt.tight_layout()
    print("Saving files...")

    os.makedirs("./figures", exist_ok=True)

    fig.savefig(
        "./figures/comb_ann_platform_groups_bubble_chart.svg",
        format="svg",
        bbox_inches="tight",
        transparent=True,
    )
    fig.savefig(
        "./figures/comb_ann_platform_groups_bubble_chart.png",
        format="png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


# Run the plotting function
create_depmap_style_bubble_chart(
    comb_ann,
    rna_batch_palette,
    spacing_intra_platform=1.0,
    spacing_inter_platform=3.0,
    figsize=(5.5, 5.5),
    dot_size_scale=1,
)

In [ ]:
def plot_palette(palette, title=None, file="palette.svg"):
    """
    Display a color palette as a legend-style swatch figure.

    Parameters
    ----------
    palette : dict
        Mapping of label → color (any matplotlib-compatible color format).
    title : str, optional
        Figure title shown above the swatches.
    """
    fig, ax = plt.subplots(figsize=(4, max(2.5, len(palette) * 0.1)))
    ax.axis("off")
    handles = [
        mpatches.Patch(color=color, label=str(name)) for name, color in palette.items()
    ]
    ax.legend(
        handles=handles,
        loc="center",
        ncol=1,
        fontsize=7.5,
        frameon=False,
        labelspacing=0.1,
        title=title,
    )
    plt.tight_layout()
    plt.show()
    fig.savefig(file, format="svg", bbox_inches="tight", transparent=True)


# plot_palette(rna_batch_palette, title="RNA_BATCH", file="./figures/RNA_BATCH_palette.svg")

plot_palette(
    lymphoma_ontogeny_palette,
    title="Biology group",
    file="./figures/lymphoma_ontogeny_palette.svg",
)

In [ ]:
def create_biology_bubble_chart(
    df, palette, spacing_groups=5.0, figsize=(10, 5), dot_size_scale=0.55
):

    # 1. Single-Tier Circlify setup
    group_counts = df["Diagnosis_cell_type_unified"].value_counts()
    pack_data = [{"id": bio, "datum": count} for bio, count in group_counts.items()]

    circles = circ.circlify(pack_data, show_enclosure=False)

    nodes = []
    HEX_SPACING = 1.2

    for bio, count in group_counts.items():
        N = count
        R_tight = math.sqrt((N * 0.866 * (HEX_SPACING**2)) / math.pi)

        cx, cy = 0.0, 0.0
        for c in circles:
            if c.ex["id"] == bio:
                cx, cy = c.x * 100, c.y * 100
                break

        nodes.append({"id": bio, "N": N, "R_tight": R_tight, "x": cx, "y": cy})

    # 2. Physics Engine (Flat single-tier layout)
    print(f"Running physics layout (Spacing: {spacing_groups})...")
    for _ in range(400):
        # A. Gravity towards center (0,0)
        for n in nodes:
            n["x"] -= n["x"] * 0.03
            n["y"] -= n["y"] * 0.03

        # B. Collision / Avoid overlap
        for i in range(len(nodes)):
            for j in range(i + 1, len(nodes)):
                n1, n2 = nodes[i], nodes[j]
                dx = n1["x"] - n2["x"]
                dy = n1["y"] - n2["y"]
                dist = math.hypot(dx, dy)

                # Minimum distance incorporates the single spacing parameter
                min_dist = n1["R_tight"] + n2["R_tight"] + spacing_groups

                if dist < min_dist and dist > 0:
                    overlap = min_dist - dist
                    push_x = (dx / dist) * overlap * 0.5
                    push_y = (dy / dist) * overlap * 0.5
                    n1["x"] += push_x
                    n1["y"] += push_y
                    n2["x"] -= push_x
                    n2["y"] -= push_y

    # =========================================================================
    # 3. DOT PLACEMENT & EDGE MORPHING (Continents & Seas)
    # =========================================================================

    cluster_candidates = {}
    for i, n in enumerate(nodes):
        candidates = generate_candidate_hex_points(
            n["N"], extra_factor=3.0, spacing=HEX_SPACING
        )

        # Unique rotation per biology group
        angle_degrees = (i * 43) % 360
        theta = math.radians(angle_degrees)
        cos_t, sin_t = math.cos(theta), math.sin(theta)

        transformed_pool = []
        for px, py in candidates:
            nx = px * cos_t - py * sin_t + n["x"]
            ny = px * sin_t + py * cos_t + n["y"]
            transformed_pool.append((nx, ny))

        cluster_candidates[i] = transformed_pool

    collision_radius = HEX_SPACING * 0.95
    spatial_hash = {}

    def check_collision(x, y, cid):
        cx, cy = int(x / collision_radius), int(y / collision_radius)
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                cell = (cx + dx, cy + dy)
                if cell in spatial_hash:
                    for px, py, pcid in spatial_hash[cell]:
                        if pcid != cid:
                            if math.hypot(px - x, py - y) < collision_radius:
                                return True
        return False

    all_x, all_y, all_c = [], [], []
    placed_counts = {i: 0 for i in range(len(nodes))}
    candidate_indices = {i: 0 for i in range(len(nodes))}

    while any(placed_counts[i] < nodes[i]["N"] for i in range(len(nodes))):
        for i, n in enumerate(nodes):
            if placed_counts[i] < n["N"]:
                placed = False

                while candidate_indices[i] < len(cluster_candidates[i]) and not placed:
                    px, py = cluster_candidates[i][candidate_indices[i]]
                    candidate_indices[i] += 1

                    if not check_collision(px, py, i):
                        all_x.append(px)
                        all_y.append(py)
                        all_c.append(palette.get(n["id"], "#cccccc"))
                        placed_counts[i] += 1

                        cx, cy = int(px / collision_radius), int(py / collision_radius)
                        if (cx, cy) not in spatial_hash:
                            spatial_hash[(cx, cy)] = []
                        spatial_hash[(cx, cy)].append((px, py, i))

                        placed = True

    # =========================================================================
    # 4. PLOTTING
    # =========================================================================

    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    ax.set_aspect("equal")
    ax.axis("off")

    min_x, max_x = min(all_x), max(all_x)
    min_y, max_y = min(all_y), max(all_y)
    pad = (max_x - min_x) * 0.05
    ax.set_xlim(min_x - pad, max_x + pad)
    ax.set_ylim(min_y - pad, max_y + pad)

    fig_width_inches = fig.get_size_inches()[0]
    ax_width_points = (fig_width_inches * 0.95) * 72
    data_width = (max_x + pad) - (min_x - pad)

    pts_per_unit = ax_width_points / data_width
    marker_size = (dot_size_scale * pts_per_unit) ** 2

    # Draw the data points
    ax.scatter(
        all_x, all_y, s=marker_size, c=all_c, edgecolors="none", alpha=0.9, zorder=1
    )

    # Render Labels & Halos
    for n in nodes:
        if n["N"] > 0:
            current_n = n["N"]
            bio_text = n["id"].replace("_", "\n") + f"\n({str(current_n)})"
            t_x = n["x"]
            t_y = n["y"]

            # Halo radius based on cluster size
            halo_radius = n["R_tight"] * 0.70
            draw_gradient_halo(ax, t_x, t_y, halo_radius, zorder=8.5)

            # Dynamic font size based on number of dots
            # font_size = max(8, math.sqrt(n['N']) * 0.45)

            ax.text(
                t_x,
                t_y,
                bio_text,
                ha="center",
                va="center",
                fontsize=7.5,
                # weight='bold',
                zorder=9,
            )

    plt.tight_layout()
    print("Saving files...")

    os.makedirs("./figures", exist_ok=True)

    fig.savefig(
        "./figures/comb_ann_biology_bubble_chart.svg",
        format="svg",
        bbox_inches="tight",
        transparent=True,
    )
    fig.savefig(
        "./figures/comb_ann_biology_bubble_chart.png",
        format="png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()


# Run the plotting function (example usage)
create_biology_bubble_chart(
    comb_ann,
    lymphoma_ontogeny_palette,
    spacing_groups=3.0,
    figsize=(4, 5.5),
    dot_size_scale=1,
)

## Sankey plot of samples by batch removal strategies

In [ ]:
RARE_GROUPS: list[str] = [
    "Extranodal_Marginal_Zone_Lymphoma",
    "Double_Hit_Lymphoma",
    "Mantle_Cell_Lymphoma",
    "Chronic_Lymphocytic_Leukemia",
    "Other",
]
NORMAL_GROUPS: list[str] = [
    "Memory",
    "Centroblast",
    "Centrocyte",
    "Bone_marrow_CD19+",
    "Plasma",
    "Naive",
    "GC",
    "B_cells",
    "MZ",
    "Plasmablast",
    "Immature",
]
BAD_BATCHES_A: list[str] = [
    "GPL14951_FFPE_Unknown",
    "GPL8432_FFPE_Unknown",
    "GPL13938_FFPE_Unknown",
]
BAD_COHORTS_A: list[str] = ["SOM"]
BAD_BATCHES_B: list[str] = BAD_BATCHES_A + [
    "GPL6244_FFPE_Unknown",
    "GPL17586_FF_Unknown",
    "GPL20188_FF_Unknown",
    "GPL23541_FF_Unknown",
    "GPL887_FFPE_Unknown",
    "GPL26356_FF_Unknown",
    "GPL17047_FF_Unknown",
    "GPL6244_FF_Unknown",
]
BAD_COHORTS_B: list[str] = ["SOM"]
MIN_BATCH_SIZE: int = 50

# Explicit set of all Affymetrix microarray RNA_BATCH labels present in the dataset.
# Used by build_filter_strategies for H_affymetrix_extended.
# Excludes GPL14951_FFPE_Unknown (Illumina HumanHT-12 V3.0, not Affymetrix).
_AFFY_EXT_BATCHES: frozenset[str] = frozenset(
    {
        "GPL570_Unknown_Unknown",
        "GPL570_FF_Unknown",
        "GPL570_FFPE_Unknown",
        "GPL96+97_FF_Unknown",
        "GPL96_FF_Unknown",
        "GPL96_FFPE_Unknown",
        "GPL6244_FF_Unknown",
        "GPL6244_FFPE_Unknown",
        "GPL20188_FF_Unknown",
        "GPL16686_FF_Unknown",
        "GPL13158_FF_Unknown",
        "GPL17586_FF_Unknown",
        "GPL23541_FF_Unknown",
        "GPL17047_FF_Unknown",
        "GPL10739_FF_Unknown",
        "GPL26356_FF_Unknown",
    }
)


BIO_COL = "Diagnosis_cell_type_unified"
BATCH_COL = "RNA_BATCH"
S3_BUCKET = "$FL_S3_BUCKET"
S3_PREFIX = "FL_batch_correction"


def build_filter_strategies(
    comb_ann: pd.DataFrame,
    comb_exp: pd.DataFrame,
    cache_path: str = "/tmp/bench_filter_strategies.pkl",
) -> dict[str, pd.DataFrame]:
    """
    Build all 14 filter strategies from annotation and expression data.

    E1–E3 require iterative PCA-based outlier detection and take extra time.
    Results are cached to cache_path so parallel workers reuse them.

    Parameters
    ----------
    comb_ann
        Full annotation DataFrame.
    comb_exp
        Full expression matrix.
    cache_path
        Pickle cache path. Pass empty string to disable caching.

    Returns
    -------
    Dict mapping strategy name → filtered annotation DataFrame (14 entries).
    """
    if cache_path and os.path.exists(cache_path):
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    ann_base = comb_ann[~comb_ann[BIO_COL].isin(RARE_GROUPS)].copy()

    rnaseq_batches = [b for b in ann_base[BATCH_COL].unique() if b.startswith("RNASeq")]
    microarray_batches = [
        b for b in ann_base[BATCH_COL].unique() if not b.startswith("RNASeq")
    ]
    affymetrix_batches = [
        b for b in ann_base[BATCH_COL].unique() if b.startswith("GPL570")
    ]
    ff_batches = [b for b in ann_base[BATCH_COL].unique() if "_FF_" in b]
    ffpe_batches = [b for b in ann_base[BATCH_COL].unique() if "_FFPE_" in b]
    malignant_groups = [g for g in ann_base[BIO_COL].unique() if g not in NORMAL_GROUPS]

    def _f(**kw: object) -> pd.DataFrame:
        return apply_filter(ann_base, **kw)  # type: ignore[arg-type]

    ann_S0 = ann_base.copy()
    ann_A = _f(bad_batches=BAD_BATCHES_A, bad_cohorts=BAD_COHORTS_A)
    ann_B = _f(bad_batches=BAD_BATCHES_B, bad_cohorts=BAD_COHORTS_B)
    ann_C = _f(keep_batches=rnaseq_batches)
    ann_D = _f(keep_groups=malignant_groups)
    ann_F = _f(keep_batches=microarray_batches)
    ann_G = _f(keep_batches=affymetrix_batches)

    _batch_counts = ann_base[BATCH_COL].value_counts()
    _large_batches = _batch_counts[_batch_counts >= MIN_BATCH_SIZE].index.tolist()
    ann_I = _f(keep_batches=_large_batches)
    ann_J = _f(keep_batches=ff_batches)
    ann_K = _f(keep_batches=ffpe_batches)

    exp_A, _ = prepare_dataset(ann_A, comb_exp)
    exp_A = log_transform_by_cohort(exp_A, ann_A.loc[exp_A.index])
    out_E1 = identify_outlier_batches(exp_A, ann_A)
    ann_E1 = _f(bad_batches=BAD_BATCHES_A + out_E1, bad_cohorts=BAD_COHORTS_A)

    exp_E1, _ = prepare_dataset(ann_E1, comb_exp)
    exp_E1 = log_transform_by_cohort(exp_E1, ann_E1.loc[exp_E1.index])
    out_E2 = identify_outlier_batches(exp_E1, ann_E1)
    ann_E2 = apply_filter(ann_E1, bad_batches=out_E2)

    exp_E2, _ = prepare_dataset(ann_E2, comb_exp)
    exp_E2 = log_transform_by_cohort(exp_E2, ann_E2.loc[exp_E2.index])
    out_E3 = identify_outlier_batches(exp_E2, ann_E2)
    ann_E3 = apply_filter(ann_E2, bad_batches=out_E3)

    strategies = {
        "S0_no_removal": ann_S0,
        "A_confirmed_bad": ann_A,
        "B_extended_bad": ann_B,
        "C_rnaseq_only": ann_C,
        "D_malignant_only": ann_D,
        "E1_iterative_r1": ann_E1,
        "E2_iterative_r2": ann_E2,
        "E3_iterative_r3": ann_E3,
        "F_microarray_only": ann_F,
        "G_affymetrix_only": ann_G,
        "H_affymetrix_extended": ann_base[
            ann_base[BATCH_COL].isin(_AFFY_EXT_BATCHES)
        ].copy(),
        "I_rare_batches_removed": ann_I,
        "J_ff_only": ann_J,
        "K_ffpe_only": ann_K,
    }

    if cache_path:
        with open(cache_path, "wb") as f:
            pickle.dump(strategies, f, protocol=4)

    return strategies


def apply_filter(
    ann: pd.DataFrame,
    bad_batches: list[str] | tuple[str, ...] = (),
    bad_cohorts: list[str] | tuple[str, ...] = (),
    keep_batches: list[str] | None = None,
    keep_groups: list[str] | None = None,
    bio_col: str = BIO_COL,
) -> pd.DataFrame:
    """
    Apply sample-level filters to an annotation DataFrame.

    Parameters
    ----------
    ann
        Annotation DataFrame to filter.
    bad_batches
        RNA_BATCH values to exclude.
    bad_cohorts
        COHORT_LABEL values to exclude.
    keep_batches
        If provided, only samples in these RNA_BATCH values are retained.
    keep_groups
        If provided, only samples in these bio_col groups are retained.
    bio_col
        Column for group filtering.

    Returns
    -------
    Filtered annotation DataFrame.
    """
    mask = pd.Series(True, index=ann.index)
    if bad_batches:
        mask &= ~ann[BATCH_COL].isin(bad_batches)
    if bad_cohorts:
        mask &= ~ann["COHORT_LABEL"].isin(bad_cohorts)
    if keep_batches is not None:
        mask &= ann[BATCH_COL].isin(keep_batches)
    if keep_groups is not None:
        mask &= ann[bio_col].isin(keep_groups)
    return ann[mask]


def prepare_dataset(
    ann_filter: pd.DataFrame,
    exp_full: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Align expression to annotation with strict full-coverage (no NAs).

    Parameters
    ----------
    ann_filter
        Filtered annotation DataFrame.
    exp_full
        Full expression matrix (samples × genes).

    Returns
    -------
    Tuple of (exp, ann) with NA-free expression.
    """
    common = ann_filter.index.intersection(exp_full.index)
    exp = exp_full.loc[common].dropna(axis=1)
    exp = exp.loc[~exp.index.duplicated(keep="first")]
    ann = ann_filter.loc[exp.index]
    return exp, ann


def log_transform_by_cohort(
    exp_df: pd.DataFrame,
    ann_df: pd.DataFrame,
    batch_col: str = "COHORT_LABEL",
    threshold: float = 30,
) -> pd.DataFrame:
    """
    Apply log2(x+1) transform per cohort only when max value exceeds threshold.

    Parameters
    ----------
    exp_df
        Expression matrix (samples × genes).
    ann_df
        Annotation with batch_col column.
    batch_col
        Column used to group samples (COHORT_LABEL, not RNA_BATCH).
    threshold
        Cohorts with max > threshold are log2(x+1) transformed.

    Returns
    -------
    Log2-transformed expression matrix.
    """
    batches = ann_df[batch_col].value_counts().index
    df_list = []
    for batch in batches:
        samples = ann_df.index[ann_df[batch_col] == batch]
        exp_local = exp_df.reindex(samples)
        if exp_local.max().max() > threshold:
            exp_local = np.log2(exp_local + 1)
        df_list.append(exp_local)
    return pd.concat(df_list, axis=0)


def log_transform_by_cohort(
    exp_df: pd.DataFrame,
    ann_df: pd.DataFrame,
    batch_col: str = "COHORT_LABEL",
    threshold: float = 30,
) -> pd.DataFrame:
    """
    Apply log2(x+1) transform per cohort only when max value exceeds threshold.

    Parameters
    ----------
    exp_df
        Expression matrix (samples × genes).
    ann_df
        Annotation with batch_col column.
    batch_col
        Column used to group samples (COHORT_LABEL, not RNA_BATCH).
    threshold
        Cohorts with max > threshold are log2(x+1) transformed.

    Returns
    -------
    Log2-transformed expression matrix.
    """
    batches = ann_df[batch_col].value_counts().index
    df_list = []
    for batch in batches:
        samples = ann_df.index[ann_df[batch_col] == batch]
        exp_local = exp_df.reindex(samples)
        if exp_local.max().max() > threshold:
            exp_local = np.log2(exp_local + 1)
        df_list.append(exp_local)
    return pd.concat(df_list, axis=0)


# ── Batch-effect metrics ──────────────────────────────────────────────────────
def pca_variance_explained_by_batch(
    exp_df: pd.DataFrame,
    batch_series: pd.Series,
    n_components: int = 10,
) -> float:
    """
    Mean fraction of PCA variance explained by batch (one-way ANOVA R²).

    Parameters
    ----------
    exp_df
        Expression matrix (samples × genes).
    batch_series
        Batch labels indexed by sample ID.
    n_components
        Number of PCs to average over.

    Returns
    -------
    Mean R² across first n_components PCs.
    """
    pca = PCA(n_components=n_components)
    coords = pca.fit_transform(StandardScaler().fit_transform(exp_df.fillna(0)))
    groups = batch_series.reindex(exp_df.index).fillna("NA")
    r2_list = []
    for pc in range(n_components):
        vals = coords[:, pc]
        grand_mean = vals.mean()
        ss_between = sum(
            len(vals[groups == g]) * (vals[groups == g].mean() - grand_mean) ** 2
            for g in groups.unique()
            if (groups == g).sum() > 1
        )
        ss_total = ((vals - grand_mean) ** 2).sum()
        r2_list.append(ss_between / ss_total if ss_total > 0 else 0)
    return float(np.mean(r2_list))


def identify_outlier_batches(
    exp_df: pd.DataFrame,
    ann_df: pd.DataFrame,
    batch_col: str = BATCH_COL,
    n_pcs: int = 2,
    n_outliers: int = 1,
    min_batch_size: int = 20,
) -> list[str]:
    """
    Identify the batches whose PCA centroid is furthest from the global centroid.

    Parameters
    ----------
    exp_df
        Expression matrix (samples × genes).
    ann_df
        Annotation with batch_col.
    batch_col
        Column for batch grouping.
    n_pcs
        Number of PCs for centroid calculation.
    n_outliers
        Number of outlier batches to return.
    min_batch_size
        Batches smaller than this are excluded from outlier detection.

    Returns
    -------
    List of outlier batch names, length <= n_outliers.
    """
    common = exp_df.index.intersection(ann_df.index)
    X = StandardScaler().fit_transform(exp_df.loc[common].fillna(0))
    coords = PCA(n_components=n_pcs).fit_transform(X)
    coords_df = pd.DataFrame(coords, index=common)
    batches = ann_df.loc[common, batch_col].fillna("NA")
    global_centroid = coords_df.mean(axis=0).values

    batch_centroids: dict[str, float] = {}
    for b in batches.unique():
        idx = batches[batches == b].index
        if len(idx) < min_batch_size:
            continue
        batch_centroids[b] = float(
            np.linalg.norm(coords_df.loc[idx].mean(axis=0).values - global_centroid)
        )

    ranked = sorted(batch_centroids, key=batch_centroids.get, reverse=True)  # type: ignore[arg-type]
    return ranked[:n_outliers]

In [ ]:
strategies_dict = build_filter_strategies(comb_ann, comb_exp)

In [ ]:
samples_lists = {}
for strat in strategies_dict.keys():
    samples_lists.update({strat: strategies_dict[strat].index.to_list()})
    print(strat, len(samples_lists[strat]))

counts = {strat: len(samples_lists[strat]) for strat in strategies_dict.keys()}

In [ ]:
samples_lists

### Sankey plot itself

In [ ]:
generate_radial_sankey()

In [ ]:
# =============================================================================
# 2. GEOMETRY HELPERS
# =============================================================================
def transform(x, y, theta_deg):
    """Converts local ribbon coordinates into global radial (Cartesian) space."""
    theta = math.radians(theta_deg)
    X = y * math.cos(theta) - x * math.sin(theta)
    Y = y * math.sin(theta) + x * math.cos(theta)
    return X, Y


def draw_curved_ribbon(ax, y1, theta1, y2, theta2, width, color, R_OUT, alpha=0.65):
    """Draws a fluid cubic Bezier ribbon in radial space."""
    # Ensure curvature ignores the hidden base segment underneath the ring
    d = (y2 - max(y1, R_OUT)) * 0.5

    xL = -width / 2.0
    xR = width / 2.0

    p0L = transform(xL, y1, theta1)
    p1L = transform(xL, max(y1, R_OUT) + d, theta1)
    p2L = transform(xL, y2 - d, theta2)
    p3L = transform(xL, y2, theta2)

    p0R = transform(xR, y1, theta1)
    p1R = transform(xR, max(y1, R_OUT) + d, theta1)
    p2R = transform(xR, y2 - d, theta2)
    p3R = transform(xR, y2, theta2)

    verts = [p0L, p1L, p2L, p3L, p3R, p2R, p1R, p0R, p0L]
    codes = [
        Path.MOVETO,
        Path.CURVE4,
        Path.CURVE4,
        Path.CURVE4,
        Path.LINETO,
        Path.CURVE4,
        Path.CURVE4,
        Path.CURVE4,
        Path.CLOSEPOLY,
    ]

    # zorder=2 ensures ribbons are drawn underneath the central ring
    patch = patches.PathPatch(
        Path(verts, codes), facecolor=color, edgecolor="none", alpha=alpha, zorder=2
    )
    ax.add_patch(patch)


def darken_color(color, factor=0.75):
    """Returns a darker version of the palette color for the terminal boxes."""
    try:
        rgb = mcolors.to_rgb(color)
        return tuple(c * factor for c in rgb)
    except:
        return color


def get_span(w, R):
    """Calculates the exact angular span (in degrees) of a ribbon at a given radius."""
    val = (w / 2.0) / R
    val = max(-1.0, min(1.0, val))  # Clamp to prevent domain errors
    return math.degrees(math.asin(val)) * 2.0


# =============================================================================
# 3. MAIN PLOTTING FUNCTION
# =============================================================================
def create_circos_sankey(figsize=(10, 10), fontsize=7.5, palette=strat_pal):
    fig, ax = plt.subplots(figsize=figsize, dpi=300)
    ax.set_aspect("equal")
    ax.axis("off")

    # --- Spatial Parameters ---
    SCALE = 0.0013  # Mapping count to visual width
    R_IN = 5.5  # Inner radius of the ring
    R_OUT = 7.5  # Outer radius of the ring
    L1, L2, L3, L4 = 16, 26, 36, 46  # Scaled radial distances for 10x10 figsize
    RECT_H = 1.0  # Height of the strategy rectangles

    # --- Define Node Layout (Theta angles & Radii) ---
    nodes = {
        "Ring_J": {"y": R_OUT, "theta": 12},
        "Ring_K": {"y": R_OUT, "theta": 48},
        "Ring_C": {"y": R_OUT, "theta": 80},
        "Ring_F": {"y": R_OUT, "theta": 124},
        "Ring_D": {"y": R_OUT, "theta": 186},
        "Ring_I": {"y": R_OUT, "theta": 258},
        "Ring_A": {"y": R_OUT, "theta": 328},
        "J_ff_only": {"y": L1, "theta": 12},
        "K_ffpe_only": {"y": L1, "theta": 48},
        "C_rnaseq_only": {"y": L1, "theta": 80},
        "F_microarray_only": {"y": L1, "theta": 124},
        "D_malignant_only": {"y": L1, "theta": 186},
        "I_rare_batches_removed": {"y": L1, "theta": 258},
        "A_confirmed_bad": {"y": L1, "theta": 328},
        "H_affymetrix_extended": {"y": L2, "theta": 124},
        "G_affymetrix_only": {"y": L3, "theta": 124},
        "B_extended_bad": {"y": L2, "theta": 305},
        "E1_iterative_r1": {"y": L2, "theta": 350},
        "E2_iterative_r2": {"y": L3, "theta": 350},
        "E3_iterative_r3": {"y": L4, "theta": 350},
    }

    # --- Dynamic Arcs (Perfectly Embracing the Ribbons) ---
    arcs = [
        {
            "label": "By biomaterial",
            "start": 12 - get_span(counts["J_ff_only"] * SCALE, R_OUT) / 2 - 1.5,
            "end": 48 + get_span(counts["K_ffpe_only"] * SCALE, R_OUT) / 2 + 1.5,
            "text_angle": 30,
        },  # Centered perfectly in the gap between J and K
        {
            "label": "By platform",
            "start": 80 - get_span(counts["C_rnaseq_only"] * SCALE, R_OUT) / 2 - 1.5,
            "end": 124 + get_span(counts["F_microarray_only"] * SCALE, R_OUT) / 2 + 1.5,
            "text_angle": 95,
        },  # Centered perfectly in the gap between C and F
        {
            "label": "By biology",
            "start": 186
            - get_span(counts["D_malignant_only"] * SCALE, R_OUT) / 2
            - 1.5,
            "end": 186 + get_span(counts["D_malignant_only"] * SCALE, R_OUT) / 2 + 1.5,
            "text_angle": 186,
        },  # Only one branch, so text goes directly over the ribbon
        {
            "label": "Bad batches\nremoval",
            "start": 258
            - get_span(counts["I_rare_batches_removed"] * SCALE, R_OUT) / 2
            - 1.5,
            "end": 328 + get_span(counts["A_confirmed_bad"] * SCALE, R_OUT) / 2 + 1.5,
            "text_angle": 297,
        },  # Centered perfectly in the gap between I and A
    ]

    # 1. Draw Arcs and Arc Text
    ring_color = palette["S0_no_removal"]
    for arc in arcs:
        # zorder=3 covers the flat ribbon bases beneath perfectly
        wedge = patches.Wedge(
            (0, 0),
            R_OUT,
            arc["start"],
            arc["end"],
            width=(R_OUT - R_IN),
            facecolor=ring_color,
            edgecolor="white",
            linewidth=1.5,
            alpha=1.0,
            zorder=3,
        )
        ax.add_patch(wedge)

        th = arc["text_angle"] % 360
        ha, rot = ("right", th - 180) if 90 < th < 270 else ("left", th)

        txt_x, txt_y = transform(0, R_OUT + 0.5, th)
        ax.text(
            txt_x,
            txt_y,
            arc["label"],
            ha=ha,
            va="center",
            rotation=rot,
            fontsize=fontsize + 2,
            weight="bold",
            color="#333333",
            zorder=15,
        )

    # --- Define Flows (Parent -> Child) ---
    flows = [
        ("Ring_J", "J_ff_only"),
        ("Ring_K", "K_ffpe_only"),
        ("Ring_C", "C_rnaseq_only"),
        ("Ring_F", "F_microarray_only"),
        ("F_microarray_only", "H_affymetrix_extended"),
        ("H_affymetrix_extended", "G_affymetrix_only"),
        ("Ring_D", "D_malignant_only"),
        ("Ring_I", "I_rare_batches_removed"),
        ("Ring_A", "A_confirmed_bad"),
        ("A_confirmed_bad", "B_extended_bad"),
        ("A_confirmed_bad", "E1_iterative_r1"),
        ("E1_iterative_r1", "E2_iterative_r2"),
        ("E2_iterative_r2", "E3_iterative_r3"),
    ]

    # 2. Draw Ribbons
    for parent, child in flows:
        n1, n2 = nodes[parent], nodes[child]
        w = counts[child] * SCALE

        y1_draw = n1["y"] + (RECT_H / 2)
        y2_draw = n2["y"] - (RECT_H / 2)

        # Ribbons start beneath the ring for a seamless flush look
        if parent.startswith("Ring_"):
            y1_draw = R_IN

        draw_curved_ribbon(
            ax,
            y1_draw,
            n1["theta"],
            y2_draw,
            n2["theta"],
            width=w,
            color=palette[child],
            R_OUT=R_OUT,
            alpha=0.65,
        )

    # 3. Draw Strategy Rectangles & Horizontal Text
    for name, n in nodes.items():
        if name.startswith("Ring_"):
            continue

        w = counts[name] * SCALE
        theta = n["theta"]

        corners_local = [
            (-w / 2, n["y"] - RECT_H / 2),
            (w / 2, n["y"] - RECT_H / 2),
            (w / 2, n["y"] + RECT_H / 2),
            (-w / 2, n["y"] + RECT_H / 2),
        ]
        corners_global = [transform(lx, ly, theta) for lx, ly in corners_local]

        rect_color = darken_color(palette[name], factor=0.70)
        rect = patches.Polygon(
            corners_global, facecolor=rect_color, edgecolor="none", zorder=4
        )
        ax.add_patch(rect)

        # Smart Horizontal Text Placement by Quadrant
        th = theta % 360
        X, Y = transform(0, n["y"], theta)

        dx = (w / 2) * abs(math.sin(math.radians(th)))
        dy = (w / 2) * abs(math.cos(math.radians(th)))
        pad = 0.8

        if 45 <= th <= 135:  # Top Quadrant
            tx, ty = X, Y + dy + pad
            ha, va = "center", "bottom"
        elif 225 <= th <= 315:  # Bottom Quadrant
            tx, ty = X, Y - dy - pad
            ha, va = "center", "top"
        elif th < 45 or th > 315:  # Right Quadrant
            tx, ty = X + dx + pad, Y
            ha, va = "left", "center"
        else:  # Left Quadrant
            tx, ty = X - dx - pad, Y
            ha, va = "right", "center"

        label = f"{name.replace('_', ' ')}\n(N={counts[name]})"
        ax.text(
            tx,
            ty,
            label,
            ha=ha,
            va=va,
            rotation=0,
            fontsize=fontsize,
            color="#111111",
            zorder=10,
        )

    # --- Finalize and Save ---
    max_radius = L4 + 14
    ax.set_xlim(-max_radius, max_radius)
    ax.set_ylim(-max_radius, max_radius)

    os.makedirs("./figures", exist_ok=True)
    out_svg = "./figures/circos_sankey_strategies.svg"
    out_png = "./figures/circos_sankey_strategies.png"

    fig.savefig(out_svg, format="svg", bbox_inches="tight", transparent=True)
    fig.savefig(out_png, format="png", dpi=300, bbox_inches="tight")
    print(f"Successfully saved to {out_svg}")
    plt.show()

In [ ]:
create_circos_sankey(figsize=(10, 10), fontsize=7.5, palette=strat_pal)

## Barplot of BATCH vs BIOLOGY

In [ ]:
s3 = boto3.client("s3")
body = s3.get_object(
    Bucket="$FL_S3_BUCKET",
    Key="FL_batch_correction/prepared/S0_no_removal__knn__ann.tsv.gz",
)["Body"].read()

comb_ann = pd.read_csv(io.BytesIO(gzip.decompress(body)), sep="\t", index_col=0)

batch_to_plaform_group = {
    "RNASeq_FF_PolyA": "Illumina NGS",
    "GPL570_Unknown_Unknown": "Affymetrix Microarray",
    "GPL570_FF_Unknown": "Affymetrix Microarray",
    "RNASeq_FFPE_Exome_capture": "Illumina NGS",
    "GPL14951_FFPE_Unknown": "Illumina Microarray",
    "GPL570_FFPE_Unknown": "Affymetrix Microarray",
    "GPL96+97_FF_Unknown": "Affymetrix Microarray",
    "GPL13938_FFPE_Unknown": "Illumina Microarray",
    "GPL96_FF_Unknown": "Affymetrix Microarray",
    "GPL6244_FF_Unknown": "Affymetrix Microarray",
    "GPL8432_FFPE_Unknown": "Illumina Microarray",
    "RNASeq_FF_rRNADepletion": "Illumina NGS",
    "GPL96_FFPE_Unknown": "Affymetrix Microarray",
    "RNASeq_FF_Unknown": "Illumina NGS",
    "GPL20188_FF_Unknown": "Affymetrix Microarray",
    "GPL16686_FF_Unknown": "Affymetrix Microarray",
    "GPL13158_FF_Unknown": "Affymetrix Microarray",
    "GPL17586_FF_Unknown": "Affymetrix Microarray",
    "GPL1708_FF_Unknown": "Agilent Microarray",
    "GPL23541_FF_Unknown": "Affymetrix Microarray",
    "RNASeq_FFPE_PolyA": "Illumina NGS",
    "RNASeq_FF_Total": "Illumina NGS",
    "GPL17077_FF_Unknown": "Agilent Microarray",
    "GPL887_FFPE_Unknown": "Agilent Microarray",
    "GPL26356_FF_Unknown": "Affymetrix Microarray",
    "GPL17047_FF_Unknown": "Affymetrix Microarray",
    "GPL6244_FFPE_Unknown": "Affymetrix Microarray",
    "GPL10739_FF_Unknown": "Affymetrix Microarray",
    "RNASeq_FF_Exome_capture": "Illumina NGS",
}

comb_ann["Platform_group"] = comb_ann.RNA_BATCH.map(batch_to_plaform_group)

comb_ann["Platform_group"].fillna("NA").value_counts()

comb_ann[["RNA_BATCH", "Diagnosis_cell_type_unified"]]

In [ ]:
biology_by_batch_counts = (
    comb_ann[["RNA_BATCH", "Diagnosis_cell_type_unified"]].value_counts().reset_index()
)
biology_by_batch_counts

In [ ]:
batch_order = comb_ann["RNA_BATCH"].value_counts().index

# 2. Convert the column to an ordered Categorical type
comb_ann["RNA_BATCH"] = pd.Categorical(
    comb_ann["RNA_BATCH"], categories=batch_order, ordered=True
)
fig, ax = plt.subplots(figsize=(4.9, 2.5))
sns.histplot(
    data=comb_ann,
    x="RNA_BATCH",
    hue="Diagnosis_cell_type_unified",
    palette=lymphoma_ontogeny_palette,
    linewidth=0.1,
    multiple="stack",  # stat="count",
    # discrete=True, shrink=.8
    ax=ax,
)
ax.set_yscale("log")
ax.get_legend().remove()
ax.set_xticklabels(rotation=90, labels=ax.get_xticklabels())

out_svg = "./figures/barplot_diagnoses_by_batches.svg"

fig.savefig(out_svg, format="svg", bbox_inches="tight", transparent=True)

In [ ]:
biology_order = comb_ann["Diagnosis_cell_type_unified"].value_counts().index

# 2. Convert the column to an ordered Categorical type
comb_ann["Diagnosis_cell_type_unified"] = pd.Categorical(
    comb_ann["Diagnosis_cell_type_unified"], categories=biology_order, ordered=True
)
fig, ax = plt.subplots(figsize=(4.9, 2.5))
sns.histplot(
    data=comb_ann,
    x="Diagnosis_cell_type_unified",
    hue="RNA_BATCH",
    palette=rna_batch_palette,
    linewidth=0.1,
    multiple="stack",  # stat="count",
    # discrete=True, shrink=.8
    ax=ax,
)
# ax.set_yscale('log')
ax.get_legend().remove()
ax.set_xticklabels(rotation=90, labels=ax.get_xticklabels())

out_svg = "./figures/barplot_batches_by_diagnoses.svg"

fig.savefig(out_svg, format="svg", bbox_inches="tight", transparent=True)

In [ ]:
# 1. Prepare data: cross-tabulate counts per batch and biology group
counts_df = (
    comb_ann.groupby(["RNA_BATCH", "Diagnosis_cell_type_unified"])
    .size()
    .unstack(fill_value=0)
)

# 2. Sort batches by total counts descending
batch_order = counts_df.sum(axis=1).sort_values(ascending=False).index
counts_df = counts_df.loc[batch_order]

# 3. Mathematical Transformation
# Calculate total counts per batch
totals = counts_df.sum(axis=1)

# Calculate linear proportions of each group within the batch
proportions = counts_df.div(totals, axis=0)

# Calculate the target total height (log10 of total counts)
# We replace 0 with 1 to avoid log10(0) errors; log10(1) equals a visual height of 0.
log_totals = np.log10(totals.replace(0, 1))

# Scale linear proportions so their stacked sum strictly equals log_totals
scaled_df = proportions.multiply(log_totals, axis=0)

# 4. Plotting using Pandas (which handles stacked bars cleanly)
fig, ax = plt.subplots(figsize=(4.7, 2.5))

# Ensure the colors perfectly match the column order in the dataframe
colors = [lymphoma_ontogeny_palette.get(col, "#CCCCCC") for col in scaled_df.columns]

scaled_df.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=colors,
    width=1.0,  # width=1.0 makes the bars touch, mimicking sns.histplot
    edgecolor="white",
    linewidth=0.1,
)

# 5. Format axes to perfectly masquerade as a log scale
max_log = int(np.ceil(log_totals.max()))
if max_log < 1:
    max_log = 1

ax.set_yticks(range(max_log))
# This explicitly labels the linear ticks (0, 1, 2, 3) as 10^0, 10^1, 10^2, 10^3
ax.set_yticklabels([f"$10^{i}$" for i in range(max_log)])

ax.set_ylabel("Count")
ax.set_xlabel("RNA_BATCH")

ax.get_legend().remove()
ax.tick_params(axis="x", rotation=90)

# 6. Save
import os

os.makedirs("./figures", exist_ok=True)
out_svg = "./figures/barplot_diagnoses_by_batches.svg"
fig.savefig(out_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
# 1. Prepare data: cross-tabulate counts per batch and biology group
counts_df = (
    comb_ann.groupby(["Diagnosis_cell_type_unified", "RNA_BATCH"])
    .size()
    .unstack(fill_value=0)
)

# 2. Sort batches by total counts descending
biology_order = counts_df.sum(axis=1).sort_values(ascending=False).index
counts_df = counts_df.loc[biology_order]

# 3. Mathematical Transformation
# Calculate total counts per batch
totals = counts_df.sum(axis=1)

# Calculate linear proportions of each group within the batch
proportions = counts_df.div(totals, axis=0)

# Calculate the target total height (log10 of total counts)
# We replace 0 with 1 to avoid log10(0) errors; log10(1) equals a visual height of 0.
log_totals = np.log10(totals.replace(0, 1))

# Scale linear proportions so their stacked sum strictly equals log_totals
scaled_df = proportions.multiply(log_totals, axis=0)

# 4. Plotting using Pandas (which handles stacked bars cleanly)
fig, ax = plt.subplots(figsize=(3, 2.5))

# Ensure the colors perfectly match the column order in the dataframe
colors = [rna_batch_palette.get(col, "#CCCCCC") for col in scaled_df.columns]

scaled_df.plot(
    kind="bar",
    stacked=True,
    ax=ax,
    color=colors,
    width=1.0,  # width=1.0 makes the bars touch, mimicking sns.histplot
    edgecolor="white",
    linewidth=0.1,
)

# 5. Format axes to perfectly masquerade as a log scale
max_log = int(np.ceil(log_totals.max()))
if max_log < 1:
    max_log = 1

ax.set_yticks(range(max_log))
# This explicitly labels the linear ticks (0, 1, 2, 3) as 10^0, 10^1, 10^2, 10^3
ax.set_yticklabels([f"$10^{i}$" for i in range(max_log)])

ax.set_ylabel("Count")
ax.set_xlabel("Biology group")

ax.get_legend().remove()
ax.tick_params(axis="x", rotation=90)

# 6. Save
import os

os.makedirs("./figures", exist_ok=True)
out_svg = "./figures/barplot_batches_by_diagnoses.svg"
fig.savefig(out_svg, format="svg", bbox_inches="tight", transparent=True)
plt.show()

### Barplots by cohorts: biology and batch

In [ ]:
batch_order = comb_ann["COHORT_LABEL"].value_counts().index

# 2. Convert the column to an ordered Categorical type
comb_ann["COHORT_LABEL"] = pd.Categorical(
    comb_ann["COHORT_LABEL"], categories=batch_order, ordered=True
)
fig, ax = plt.subplots(figsize=(9.5, 3.5))
sns.histplot(
    data=comb_ann,
    x="COHORT_LABEL",
    hue="Diagnosis_cell_type_unified",
    palette=lymphoma_ontogeny_palette,
    linewidth=0.1,
    multiple="stack",  # stat="count",
    # discrete=True, shrink=.8
    ax=ax,
)
ax.set_yscale("log")
ax.get_legend().remove()
ax.set_xticklabels(rotation=90, labels=ax.get_xticklabels(), fontsize=6)

out_svg = "./figures/barplot_diagnoses_by_cohorts.svg"

fig.savefig(out_svg, format="svg", bbox_inches="tight", transparent=True)

In [ ]:
batch_order = comb_ann["COHORT_LABEL"].value_counts().index

# 2. Convert the column to an ordered Categorical type
comb_ann["COHORT_LABEL"] = pd.Categorical(
    comb_ann["COHORT_LABEL"], categories=batch_order, ordered=True
)
fig, ax = plt.subplots(figsize=(9.5, 3.5))
sns.histplot(
    data=comb_ann,
    x="COHORT_LABEL",
    hue="RNA_BATCH",
    palette=rna_batch_palette,
    linewidth=0.1,
    multiple="stack",  # stat="count",
    # discrete=True, shrink=.8
    ax=ax,
)
ax.set_yscale("log")
ax.get_legend().remove()
ax.set_xticklabels(rotation=90, labels=ax.get_xticklabels(), fontsize=6)

out_svg = "./figures/barplot_batches_by_cohorts.svg"

fig.savefig(out_svg, format="svg", bbox_inches="tight", transparent=True)